# 04 · Mechanisms: heat, dryness, wind or missing rain?
**A. Observations (ERA5, fast):**
1. Anatomy of the record seasons — daily FWI and its sub-indices (ISI = wind + fine-fuel dryness; BUI/DC = deep drought) vs climatology
2. Anomaly maps (Mar–May) of Tmax, RHmin, wind, rain for the top-5 fire-weather years
3. Driver regression: which of heat / humidity / wind / pre-season rain explains the year-to-year swings in annual-max FWI

**B. Large ensembles (CMIP6 monthly data from the public Pangeo Google Cloud archive, no login):**

4. Same driver regression inside the model pools (today vs +3 °C) → do the drivers change?
5. Decomposition of the projected FWI increase into heat, humidity, wind and rain contributions
6. Circulation behind the top 5 % model years: sea-level-pressure anomaly composites (proxy for persistent anticyclones/blocking)

Runtime: part A ≈ 2 min; part B ≈ 15–30 min (reads ~10 members per model). Set `RUN_MODELS = False` to skip B.

In [ ]:
# ================= CONFIG =================
import os
BASE = os.environ.get('FWI_BASE', '/content/drive/MyDrive/FWI_India')
OBS_VERSION = 'B'
CLIM = (1991, 2020)
TOPN = 5
RUN_MODELS = True
MODELS = ['CanESM5', 'MIROC6', 'ACCESS-ESM1-5', 'MPI-ESM1-2-LR']
N_MEMBERS = 10                    # members per model read from Pangeo (raise if you have time)
GWLS_MECH = ['today', '3.0']
PANGEO_CSV = 'https://storage.googleapis.com/cmip6/pangeo-cmip6.csv'
CIRC_BOX = dict(lat=(0, 45), lon=(50, 110))
NTHREADS = 8

In [ ]:
# [COLAB-ONLY]
from google.colab import drive
drive.mount('/content/drive')
!pip -q install gcsfs cftime netCDF4 geopandas zarr

In [ ]:
import re, json, math, itertools, warnings, concurrent.futures as cf
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
RES, FIG = f'{BASE}/results', f'{BASE}/figs'
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
try:
    import geopandas as gpd
    regions_gdf = gpd.read_file(f'{BASE}/maps/regions.geojson')
except Exception:
    regions_gdf = None
regd = pd.read_csv(f'{BASE}/data/era5/era5_region_daily.csv', parse_dates=['time'])
regann = pd.read_csv(f'{BASE}/data/era5/era5_region_annual.csv', header=[0, 1, 2], index_col=0)
REGIONS = list(regd.region.unique())
print(REGIONS)

## A1. Anatomy of the record seasons (Feb–Jun)

In [ ]:
VARS = [('fwiA', 'FWI'), ('isi', 'ISI (wind + fine fuel)'), ('bui', 'BUI (build-up / drought)'), ('dc', 'Drought Code'),
        ('T', 'Tmax (°C)'), ('RH', 'RHmin (%)')]
fig, axs = plt.subplots(len(VARS), len(REGIONS), figsize=(7 * len(REGIONS), 2.2 * len(VARS)), sharex=True)
for j, reg in enumerate(REGIONS):
    d = regd[regd.region == reg].set_index('time')
    d = d[d.index.month.isin([2, 3, 4, 5, 6])]
    d['doy'] = d.index.dayofyear
    clim = d[(d.index.year >= CLIM[0]) & (d.index.year <= CLIM[1])].groupby('doy')
    top = regann[(reg, 'fwixx', OBS_VERSION)].dropna().sort_values(ascending=False).index[:3].astype(int)
    for i, (v, lab) in enumerate(VARS):
        ax = axs[i, j]
        q = clim[v].quantile([.1, .5, .9]).unstack()
        ax.fill_between(q.index, q[.1], q[.9], color='grey', alpha=.3, label=f'{CLIM[0]}–{CLIM[1]} 10–90%')
        ax.plot(q.index, q[.5], color='grey')
        for y, c in zip(top, ['r', 'darkorange', 'gold']):
            s = d[d.index.year == y]; ax.plot(s.doy, s[v], color=c, lw=1.4, label=str(y))
        ax.set_ylabel(lab, fontsize=8); ax.grid(alpha=.3)
        if i == 0:
            ax.set_title(f'{reg}: top-3 fwixx years'); ax.legend(fontsize=7)
    axs[-1, j].set_xlabel('day of year (Feb–Jun)')
plt.tight_layout(); plt.savefig(f'{FIG}/04_record_season_anatomy.png', dpi=150); plt.show()

## A2. Mar–May anomaly maps for the top-5 years of each region

In [ ]:
M = xr.open_dataset(f'{BASE}/data/era5/era5_monthly_025.nc')
mam = M.sel(month=M.month.dt.month.isin([3, 4, 5])).groupby('month.year').mean('month')
clim_mam = mam.sel(year=slice(*CLIM)).mean('year')
PANELS = [('T', 'Tmax anomaly (°C)', 'RdBu_r', 3), ('RH', 'RHmin anomaly (%)', 'BrBG', 8),
          ('W', 'wind anomaly (km/h)', 'PuOr_r', 2), ('P', 'rain anomaly (mm/month)', 'BrBG', 30), ('FWI', 'FWI anomaly', 'RdYlBu_r', 8)]
for reg in REGIONS:
    top = regann[(reg, 'fwixx', OBS_VERSION)].dropna().sort_values(ascending=False).index[:TOPN].astype(int)
    comp = mam.sel(year=[y for y in top if y in mam.year]).mean('year') - clim_mam
    fig, axs = plt.subplots(1, len(PANELS), figsize=(4.2 * len(PANELS), 3.8))
    for ax, (v, lab, cm, lim) in zip(axs, PANELS):
        comp[v].plot(ax=ax, cmap=cm, vmin=-lim, vmax=lim, cbar_kwargs=dict(label=lab, shrink=.8))
        if regions_gdf is not None:
            regions_gdf.boundary.plot(ax=ax, color='k', lw=.8)
        ax.set_title(lab.split(' (')[0])
    plt.suptitle(f'{reg}: Mar–May composite of top-{TOPN} fwixx years ({", ".join(map(str, top))}) minus {CLIM[0]}–{CLIM[1]}')
    plt.tight_layout(); plt.savefig(f'{FIG}/04_era5_composite_{reg.replace(" ", "_")}.png', dpi=150); plt.show()

## A3. Driver regression on observed years
Annual-max FWI regressed on standardised Mar–May mean Tmax, RHmin, wind and Jan–May mean rain (regional means).
**Relative importance** = LMG share of explained variance (averaged over all predictor orderings).

In [ ]:
def lmg(X, y):
    """LMG relative importance (share of R²) for each column of X."""
    p = X.shape[1]
    def r2(cols):
        if not cols:
            return 0.0
        A = np.c_[np.ones(len(y)), X[:, cols]]
        res = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
        return 1 - res.var() / y.var()
    cache = {c: r2(list(c)) for k in range(p + 1) for c in itertools.combinations(range(p), k)}
    imp = np.zeros(p)
    for perm in itertools.permutations(range(p)):
        prev = ()
        for i in perm:
            new = tuple(sorted(prev + (i,)))
            imp[i] += cache[new] - cache[prev]; prev = new
    imp /= math.factorial(p)
    return imp, cache[tuple(range(p))]

def fit_drivers(df, ycol, xcols):
    df = df.dropna(subset=[ycol] + xcols)
    X = df[xcols].values; Xs = (X - X.mean(0)) / X.std(0); y = df[ycol].values
    A = np.c_[np.ones(len(y)), Xs]; beta_std = np.linalg.lstsq(A, y, rcond=None)[0][1:]
    beta_raw = np.linalg.lstsq(np.c_[np.ones(len(y)), X], y, rcond=None)[0][1:]
    imp, R2 = lmg(Xs, y)
    return pd.DataFrame(dict(predictor=xcols, beta_per_sd=beta_std, beta_per_unit=beta_raw, rel_importance=imp / R2 if R2 > 0 else imp,
                             R2=R2, n=len(y)))

DRV = ['Tmax_MAM', 'RH_MAM', 'Wind_MAM', 'Rain_JFMAM']
obs_drv = []
for reg in REGIONS:
    d = regd[regd.region == reg].set_index('time')
    mam, jfmam = d[d.index.month.isin([3, 4, 5])], d[d.index.month <= 5]
    f = pd.DataFrame({'Tmax_MAM': mam['T'].groupby(mam.index.year).mean(),
                      'RH_MAM': mam['RH'].groupby(mam.index.year).mean(),
                      'Wind_MAM': mam['W'].groupby(mam.index.year).mean(),
                      'Rain_JFMAM': jfmam['P'].groupby(jfmam.index.year).mean()})
    f['fwixx'] = regann[(reg, 'fwixx', OBS_VERSION)].reindex(f.index)
    f = f.loc[1951:]
    r = fit_drivers(f, 'fwixx', DRV).assign(region=reg, source='ERA5', gwl='obs 1951–now'); obs_drv.append(r)
obs_drv = pd.concat(obs_drv); obs_drv.to_csv(f'{RES}/04_drivers_era5.csv', index=False)
print(obs_drv.round(3).to_string(index=False))

## B. Large ensembles — monthly drivers from Pangeo CMIP6 (Google Cloud, anonymous)

In [ ]:
BOX = {}
if regions_gdf is not None:
    for r in regions_gdf.itertuples():
        b = r.geometry.bounds; BOX[r.region] = dict(lat=(b[1], b[3]), lon=(b[0], b[2]))
else:
    BOX = {'Central India': dict(lat=(17.8, 26.9), lon=(74.0, 84.4)), 'Uttarakhand': dict(lat=(28.7, 31.5), lon=(77.5, 81.1))}
win = pd.read_csv(f'{BASE}/data/cmip6/gwl_windows_ssp585.csv')
win = win[win.in_fwi_data & win.gwl.isin(GWLS_MECH)]
fwx = xr.open_dataarray(f'{BASE}/data/cmip6/LE_fwixx_hursmin_india.nc')
W = xr.open_dataset(f'{BASE}/data/region_weights_g025.nc')
def wmean(da, w):
    w = xr.DataArray(w, dims=('lat', 'lon'), coords=dict(lat=W.lat, lon=W.lon))
    da = da.sel(lat=W.lat, lon=W.lon, method='nearest', tolerance=0.01).assign_coords(lat=W.lat, lon=W.lon)
    return (da * w).sum(('lat', 'lon')) / w.sum()
FWX = {reg: wmean(fwx, W[f'w_{reg.replace(" ", "_")}'].values).to_pandas() for reg in REGIONS}
MVARS = {'tasmax': 'Amon', 'hurs': 'Amon', 'sfcWind': 'Amon', 'pr': 'Amon', 'psl': 'Amon'}

def natural(m):
    return [int(x) for x in re.findall(r'\d+', m)]

In [ ]:
# [COLAB-ONLY]
import gcsfs
gcs = gcsfs.GCSFileSystem(token='anon')
cat = pd.read_csv(PANGEO_CSV)
cat = cat[cat.source_id.isin(MODELS) & cat.experiment_id.isin(['historical', 'ssp585']) &
          cat.variable_id.isin(list(MVARS)) & (cat.table_id == 'Amon')]
CHOSEN = {}
for m in MODELS:
    c = cat[cat.source_id == m]
    ok = c.groupby('member_id').apply(lambda g: len(set(zip(g.experiment_id, g.variable_id))) == 2 * len(MVARS))
    mems = [mm for mm in sorted(ok[ok].index, key=natural) if f'{m}|{mm}' in set(win.run)]
    CHOSEN[m] = mems[:N_MEMBERS]
    print(f'{m}: {len(mems)} members with all variables on Pangeo → using {len(CHOSEN[m])}')

def open_store(zstore):
    try:
        return xr.open_zarr(gcs.get_mapper(zstore), consolidated=True, use_cftime=True)
    except Exception:
        return xr.open_zarr(gcs.get_mapper(zstore), consolidated=False, use_cftime=True)

def lonsel(da, lo):
    lo = [(x + 360) % 360 for x in lo] if float(da.lon.min()) >= 0 else lo
    return da.sel(lon=slice(*lo))

def get_member(args):
    try:
        return _get_member(args)
    except Exception as e:
        print(f'\n  skipped {args}: {e}'); return args[0], args[1], None

def _get_member(args):
    m, mem = args
    w = win[(win.esm == m) & (win.member == mem)]
    years = sorted(set(itertools.chain(*[range(s, e + 1) for s, e in zip(w.start, w.end)])))
    out = {}
    for v in MVARS:
        parts = []
        for xp in ['historical', 'ssp585']:
            z = cat[(cat.source_id == m) & (cat.member_id == mem) & (cat.experiment_id == xp) & (cat.variable_id == v)].zstore
            if not len(z):
                continue
            da = open_store(z.iloc[-1])[v]
            da = da.rename({k: n for k, n in [('latitude', 'lat'), ('longitude', 'lon')] if k in da.dims}).sortby('lat')
            da = da.sel(time=da.time.dt.year.isin(years))
            if da.time.size:
                parts.append(da)
        da = xr.concat(parts, 'time')
        if v == 'psl':
            f = lonsel(da.sel(lat=slice(*CIRC_BOX['lat'])), CIRC_BOX['lon'])
            f = f.sel(time=f.time.dt.month.isin([3, 4, 5])).groupby('time.year').mean('time') / 100   # hPa
            out['psl_map'] = f.load()
            continue
        for reg, b in BOX.items():
            dl = float(abs(da.lat[1] - da.lat[0])) / 2; dn = float(abs(da.lon[1] - da.lon[0])) / 2
            r = lonsel(da.sel(lat=slice(b['lat'][0] - dl, b['lat'][1] + dl)), (b['lon'][0] - dn, b['lon'][1] + dn))
            r = r.weighted(np.cos(np.deg2rad(r.lat))).mean(('lat', 'lon')).load()
            months = [3, 4, 5] if v != 'pr' else [1, 2, 3, 4, 5]
            s = r.sel(time=r.time.dt.month.isin(months)).groupby('time.year').mean('time').to_pandas()
            s = s - 273.15 if v == 'tasmax' else (s * 86400 if v == 'pr' else s)
            out[(reg, v)] = s
    return m, mem, out

tasks = [(m, mem) for m in MODELS for mem in CHOSEN[m]]
MODEL_DATA = {}
if RUN_MODELS:
    with cf.ThreadPoolExecutor(NTHREADS) as ex:
        for k, (m, mem, out) in enumerate(ex.map(get_member, tasks)):
            if out is not None:
                MODEL_DATA[(m, mem)] = out
            print(f'\r{k+1}/{len(tasks)} members read', end='')
    pd.to_pickle(MODEL_DATA, f'{RES}/04_model_monthly_drivers.pkl')

In [ ]:
MODEL_DATA = pd.read_pickle(f'{RES}/04_model_monthly_drivers.pkl')
NAME = {'tasmax': 'Tmax_MAM', 'hurs': 'RH_MAM', 'sfcWind': 'Wind_MAM', 'pr': 'Rain_JFMAM'}
rows = []
for (m, mem), out in MODEL_DATA.items():
    run = f'{m}|{mem}'
    for g in GWLS_MECH:
        w = win[(win.run == run) & (win.gwl == g)]
        if w.empty:
            continue
        yrs = range(int(w.start.iloc[0]), int(w.end.iloc[0]) + 1)
        for reg in REGIONS:
            for y in yrs:
                r = dict(esm=m, member=mem, gwl=g, region=reg, year=y,
                         fwixx=FWX[reg].loc[run, y] if run in FWX[reg].index else np.nan)
                r.update({NAME[v]: out[(reg, v)].get(y, np.nan) for v in NAME})
                rows.append(r)
mdf = pd.DataFrame(rows).dropna()
mdf.to_csv(f'{RES}/04_model_driver_table.csv', index=False)
print(mdf.groupby(['esm', 'gwl', 'region']).size().unstack())

### B4. Do the drivers change between today and +3 °C?  ·  B5. What makes up the FWI increase?

In [ ]:
mod_drv, decomp = [], []
for reg in REGIONS:
    for m in MODELS:
        for g in GWLS_MECH:
            d = mdf[(mdf.region == reg) & (mdf.esm == m) & (mdf.gwl == g)]
            if len(d) > 20:
                mod_drv.append(fit_drivers(d, 'fwixx', DRV).assign(region=reg, source=m, gwl=g))
        d0 = mdf[(mdf.region == reg) & (mdf.esm == m) & (mdf.gwl == 'today')]
        d1 = mdf[(mdf.region == reg) & (mdf.esm == m) & (mdf.gwl == '3.0')]
        if len(d0) > 20 and len(d1) > 20:
            b = fit_drivers(d0, 'fwixx', DRV).set_index('predictor').beta_per_unit
            dx = d1[DRV].mean() - d0[DRV].mean()
            contrib = (b * dx).to_dict()
            decomp.append(dict(region=reg, model=m, dFWIxx_actual=d1.fwixx.mean() - d0.fwixx.mean(),
                               **{f'd_{k}': dx[k] for k in DRV}, **{f'contrib_{k}': contrib[k] for k in DRV},
                               residual=(d1.fwixx.mean() - d0.fwixx.mean()) - sum(contrib.values())))
mod_drv = pd.concat(mod_drv); mod_drv.to_csv(f'{RES}/04_drivers_models.csv', index=False)
decomp = pd.DataFrame(decomp); decomp.to_csv(f'{RES}/04_change_decomposition.csv', index=False)

allimp = pd.concat([obs_drv, mod_drv])
fig, axs = plt.subplots(1, len(REGIONS), figsize=(7 * len(REGIONS), 4))
for ax, reg in zip(np.atleast_1d(axs), REGIONS):
    p = allimp[allimp.region == reg].pivot_table(index=['source', 'gwl'], columns='predictor', values='rel_importance')[DRV]
    p.plot.barh(stacked=True, ax=ax, colormap='Set2'); ax.set_xlabel('share of explained variance (LMG)')
    ax.set_title(f'{reg}: what drives year-to-year fwixx'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIG}/04_driver_importance.png', dpi=150); plt.show()

fig, axs = plt.subplots(1, len(REGIONS), figsize=(7 * len(REGIONS), 4))
for ax, reg in zip(np.atleast_1d(axs), REGIONS):
    d = decomp[decomp.region == reg].set_index('model')
    d[[f'contrib_{k}' for k in DRV] + ['residual']].rename(columns=lambda c: c.replace('contrib_', '')).plot.bar(stacked=True, ax=ax, colormap='Set2')
    ax.scatter(range(len(d)), d.dFWIxx_actual, color='k', zorder=5, label='actual Δ fwixx')
    ax.axhline(0, color='k', lw=.6); ax.set_ylabel('Δ annual-max FWI, +3 °C minus today'); ax.set_title(reg); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIG}/04_change_decomposition.png', dpi=150); plt.show()
decomp.round(2)

### B6. Circulation behind the top 5 % model years (Mar–May sea-level-pressure anomaly, multi-model mean)
Positive anomalies over/near the region = persistent anticyclonic, subsiding, clear-sky conditions (the monthly-data proxy
for blocking; a daily blocking index would need daily 500 hPa heights — a follow-up step).

In [ ]:
target = xr.Dataset(coords=dict(lat=np.arange(CIRC_BOX['lat'][0], CIRC_BOX['lat'][1] + .1, 2.5),
                                lon=np.arange(CIRC_BOX['lon'][0], CIRC_BOX['lon'][1] + .1, 2.5)))
fig, axs = plt.subplots(len(REGIONS), len(GWLS_MECH), figsize=(6 * len(GWLS_MECH), 4.5 * len(REGIONS)))
axs = np.atleast_2d(axs)
for i, reg in enumerate(REGIONS):
    for j, g in enumerate(GWLS_MECH):
        comps = []
        for m in MODELS:
            d = mdf[(mdf.region == reg) & (mdf.esm == m) & (mdf.gwl == g)]
            if d.empty:
                continue
            thr = d.fwixx.quantile(.95)
            fields, flags = [], []
            for mem, y, fx in zip(d.member, d.year, d.fwixx):
                f = MODEL_DATA[(m, mem)]['psl_map']
                if y in f.year:
                    fields.append(f.sel(year=y).drop_vars('year')); flags.append(fx >= thr)
            F = xr.concat(fields, 'k'); flags = xr.DataArray(flags, dims='k')
            an = F.where(flags, drop=True).mean('k') - F.mean('k')
            comps.append(an.interp(lat=target.lat, lon=target.lon))
        ax = axs[i, j]
        if comps:
            xr.concat(comps, 'model').mean('model').plot.contourf(ax=ax, levels=np.linspace(-2, 2, 17), cmap='RdBu_r',
                                                                 cbar_kwargs=dict(label='hPa'))
        if regions_gdf is not None:
            regions_gdf.boundary.plot(ax=ax, color='k', lw=1)
        ax.set_title(f'{reg}: top-5% fwixx years, {g}')
plt.tight_layout(); plt.savefig(f'{FIG}/04_psl_composites.png', dpi=150); plt.show()